In [11]:
import numpy as np
from datetime import datetime, timezone
import pandas as pd
from candle_utils.candle_gen import generate_fake_candlestick_data
from PIL import ImageFont
import pytz
from functools import lru_cache

# Import stuff

In [13]:
dates, opens, highs, lows, closes, index, volume = generate_fake_candlestick_data(remove_weekends=True, interval='hourly', length=300)

print("Dates:", dates)
print("Opens:", opens)
print("Highs:", highs)
print("Lows:", lows)
print("Closes:", closes)
print("Index:", index)

Dates: [1722816000 1722819600 1722823200 1722826800 1722830400 1722834000
 1722837600 1722841200 1722844800 1722848400 1722852000 1722855600
 1722859200 1722862800 1722866400 1722870000 1722873600 1722877200
 1722880800 1722884400 1722888000 1722891600 1722895200 1722898800
 1722902400 1722906000 1722909600 1722913200 1722916800 1722920400
 1722924000 1722927600 1722931200 1722934800 1722938400 1722942000
 1722945600 1722949200 1722952800 1722956400 1722960000 1722963600
 1722967200 1722970800 1722974400 1722978000 1722981600 1722985200
 1722988800 1722992400 1722996000 1722999600 1723003200 1723006800
 1723010400 1723014000 1723017600 1723021200 1723024800 1723028400
 1723032000 1723035600 1723039200 1723042800 1723046400 1723050000
 1723053600 1723057200 1723060800 1723064400 1723068000 1723071600
 1723075200 1723078800 1723082400 1723086000 1723089600 1723093200
 1723096800 1723100400 1723104000 1723107600 1723111200 1723114800
 1723118400 1723122000 1723125600 1723129200 1723132800

In [14]:
# put these into a pandas DataFrame for easier handling
data = pd.DataFrame({
    "Index": index,
    "Date": dates,
    "Open": opens,
    "High": highs,
    "Low": lows,
    "Close": closes})
data.set_index("Index", inplace=True)

In [15]:
data

,Date,Open,High,Low,Close
Index,,,,,
0,1722816000,148.500000,152.486985,146.836262,150.000000
1,1722819600,151.120940,152.965735,149.972488,150.035207
2,1722823200,150.211239,156.323894,148.434026,152.428834
3,1722826800,153.262769,158.166219,152.595520,157.529194
4,1722830400,159.260089,161.889206,154.705345,157.264062
...,...,...,...,...,...
295,1724223600,311.777660,317.643888,305.354359,311.474760
296,1724227200,310.775084,321.511714,304.756338,318.013237
297,1724230800,318.348345,329.484140,314.519457,320.921783


# font test

In [16]:
# Load your OTF font at a specific size (e.g., 24pt)
font = ImageFont.truetype(r"C:\Users\khazy\OneDrive\Documents\DCG_Release_Tests_py11\.venv\Lib\site-packages\dearcygui\lmsans17-regular.otf", 24)

In [17]:
# Measure a single character
char = "A"
bbox = font.getbbox(char)
width, height = bbox[2] - bbox[0], bbox[3] - bbox[1]
print(f"Character '{char}' size: {width}x{height} pixels")

# Measure a string
text = "Hello, world!"
bbox = font.getbbox(text)
width, height = bbox[2] - bbox[0], bbox[3] - bbox[1]
print(f"String size: {width}x{height} pixels")

Character 'A' size: 15x17 pixels
String size: 119x20 pixels


Text size:<br>
Character 'A' size: 15x17 pixels<br>
String size: 119x20 pixels

In [19]:
# make a loop to print out the width and height of each alphabet character and number
for char in "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789":
    bbox = font.getbbox(char)
    width, height = bbox[2] - bbox[0], bbox[3] - bbox[1]
    print(f"Character '{char}' width: {width} pixels")

Character 'A' width: 15 pixels
Character 'B' width: 15 pixels
Character 'C' width: 14 pixels
Character 'D' width: 16 pixels
Character 'E' width: 13 pixels
Character 'F' width: 13 pixels
Character 'G' width: 15 pixels
Character 'H' width: 16 pixels
Character 'I' width: 6 pixels
Character 'J' width: 11 pixels
Character 'K' width: 16 pixels
Character 'L' width: 12 pixels
Character 'M' width: 20 pixels
Character 'N' width: 16 pixels
Character 'O' width: 17 pixels
Character 'P' width: 14 pixels
Character 'Q' width: 17 pixels
Character 'R' width: 15 pixels
Character 'S' width: 13 pixels
Character 'T' width: 15 pixels
Character 'U' width: 15 pixels
Character 'V' width: 15 pixels
Character 'W' width: 21 pixels
Character 'X' width: 15 pixels
Character 'Y' width: 15 pixels
Character 'Z' width: 14 pixels
Character 'a' width: 11 pixels
Character 'b' width: 12 pixels
Character 'c' width: 10 pixels
Character 'd' width: 12 pixels
Character 'e' width: 10 pixels
Character 'f' width: 8 pixels
Character 

In [21]:
#write this to a csv file called char_sizes.csv with columns "Character", "Width", "Height"
with open("char_sizes.csv", "w") as f:
    f.write("Character,Width,Height\n")
    for char in "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789":
        bbox = font.getbbox(char)
        width, height = bbox[2] - bbox[0], bbox[3] - bbox[1]
        f.write(f"{char},{width},{height}\n")

Numbers are all 11 pixels wide

# Gap remover

<span style="color: red;">This part assumes that the data is not correct and has gaps that must be found</span>

* Jupyter notebook to prototype
* Put Candle test function into notebook
* New columns needed:
    * <span style="color: green;">Time_Shift</span> - the amount that the True_Time needs to be shifted back to arrive at the Colapsed_Time. 
    * <span style="color: green;">True_Time</span> - original time from data source. Used for labels.
    * <span style="color: green;">Colapsed_Time</span> - new continuous time for plotting after gaps are removed. Used for ImGui axis.
    * Market_Hours - normal trading hours - remove all other time
    * Overnight
    * Weekends
    * Holidays
    * Market_Closure

In [13]:
# add the True_Time and Colapsed_Time columns to the DataFrame as new blank columns
data['Time_Shift'] = int(0)
data['Collapsed_Time'] = int(0)
data['True_Time'] = data['Date'].astype(int)

# reorder the columns to have True_Time and Collapsed_Time at the beginning
data = data[['True_Time', 'Collapsed_Time', 'Time_Shift'] + [col for col in data.columns if col not in ['True_Time', 'Collapsed_Time', 'Time_Shift']]]

In [14]:
data.dtypes

True_Time           int64
Collapsed_Time      int64
Time_Shift          int64
Date                int64
Open              float64
High              float64
Low               float64
Close             float64
dtype: object

In [15]:
data

,True_Time,Collapsed_Time,Time_Shift,Date,Open,High,Low,Close
Index,,,,,,,,
0,1722816000,0,0,1722816000,148.500000,152.486985,146.836262,150.000000
1,1722819600,0,0,1722819600,151.120940,152.965735,149.972488,150.035207
2,1722823200,0,0,1722823200,150.211239,156.323894,148.434026,152.428834
3,1722826800,0,0,1722826800,153.262769,158.166219,152.595520,157.529194
4,1722830400,0,0,1722830400,159.260089,161.889206,154.705345,157.264062
...,...,...,...,...,...,...,...,...
235,1723834800,0,0,1723834800,315.889426,321.240349,298.135907,304.166881
236,1723838400,0,0,1723838400,297.616982,322.119848,289.741502,318.055344
237,1723842000,0,0,1723842000,323.277861,329.806891,302.372442,306.592071


In [16]:
# make a function that detects the median delta time between values in a Series, treating values as integers (unix time)
def median_delta_time(series):
    # Ensure input is a pandas Series
    if not isinstance(series, pd.Series):
        raise ValueError("Input must be a pandas Series.")
    
    # Ensure values are integer type
    values = series.astype(int)
    
    # Calculate the time deltas
    deltas = values.diff().dropna()
    #print(deltas)
    
    # Return the median delta time
    return deltas.median()


In [17]:
median_delta_time(data['Date'])  # Assuming 'Date' is the Series of interest

np.float64(3600.0)

In [18]:
# make a function that will give any time gaps that are larger than the median delta time
def find_large_gaps(series):
    median_delta = median_delta_time(series)
    if median_delta is None:
        return []
    
    deltas = series.diff().dropna()
    large_gaps = deltas[deltas > median_delta]
    
    gaps = []
    for idx in large_gaps.index:
        start = series.iloc[idx - 1]
        stop = series.iloc[idx]
        duration = stop - start
        gaps.append({'Type': 'gap', 'start': start, 'stop': stop, 'duration': duration})
    return gaps

In [19]:
find_large_gaps(data['Date'])  # Assuming 'Date' is the Series of interest

[{'Type': 'gap',
  'start': np.int64(1723244400),
  'stop': np.int64(1723420800),
  'duration': np.int64(176400)}]

In [20]:
def chunks(gaps: list[dict[str, np.int64]], start: int, end: int):
    # The number of chunks will be the number of gaps plus 1
    '''Expected structure of gaps:
    [
        {'start': int, 
        'stop': int
        'duration': int
        },
        ...
    ]

    example: 

  [{'start': np.int64(1723244400),
  'stop': np.int64(1723420800),
  'duration': np.int64(176400)}]

    '''
    chunk_list = []
    for i in range(len(gaps) + 1):
        if i == 0:
            # First chunk: from the overall start to the start of the first gap
            chunk_list.append({'Type': 'chunk', 'start': start, 'stop': gaps[i]['start'], 'duration': gaps[i]['start'] - start})
        elif i == len(gaps):
            # Last chunk: from the end of the last gap to the overall end
            chunk_list.append({'Type': 'chunk', 'start': gaps[i - 1]['stop'], 'stop': end, 'duration': end - gaps[i - 1]['stop']})
        else:
            # Middle chunks: between the end of one gap and the start of the next gap
            chunk_list.append({'Type': 'chunk', 'start': gaps[i - 1]['stop'], 'stop': gaps[i]['start'], 'duration': gaps[i]['start'] - gaps[i - 1]['stop']})
    return chunk_list

In [21]:
chunkos=chunks(find_large_gaps(data['Date']), data['Date'].min(), data['Date'].max())
chunkos

[{'Type': 'chunk',
  'start': np.int64(1722816000),
  'stop': np.int64(1723244400),
  'duration': np.int64(428400)},
 {'Type': 'chunk',
  'start': np.int64(1723420800),
  'stop': np.int64(1723849200),
  'duration': np.int64(428400)}]

In [22]:
# for each chunk, test that the duration is correct
for chunk in chunks(find_large_gaps(data['Date']), data['Date'].min(), data['Date'].max()):
    assert chunk['duration'] == chunk['stop'] - chunk['start']
    #print diagnostics
    print(f"Chunk from {chunk['start']} to {chunk['stop']} has duration {chunk['duration']}")

Chunk from 1722816000 to 1723244400 has duration 428400
Chunk from 1723420800 to 1723849200 has duration 428400


In [23]:
def find_gaps_and_chunks(series):
    """
    Finds large gaps in a time series and returns both gaps and chunks.
    Returns:
        gaps: list of dicts with gap info
        chunks: list of dicts with chunk info
    """
    # Find gaps
    median_delta = median_delta_time(series)
    if median_delta is None:
        return [], []
    
    deltas = series.diff().dropna()
    large_gaps = deltas[deltas > median_delta]
    
    gaps = []
    for idx in large_gaps.index:
        start = series.iloc[idx - 1]
        stop = series.iloc[idx]
        duration = stop - start
        gaps.append({'Type': 'gap', 'start': start, 'stop': stop, 'duration': duration})

    # Find chunks
    start = np.int64(series.min())
    end = np.int64(series.max())
    chunk_list = []
    for i in range(len(gaps) + 1):
        if i == 0:
            # First chunk: from the overall start to the start of the first gap
            chunk_list.append({'Type': 'chunk', 'start': start, 'stop': gaps[i]['start'], 'duration': gaps[i]['start'] - start})
        elif i == len(gaps):
            # Last chunk: from the end of the last gap to the overall end
            chunk_list.append({'Type': 'chunk', 'start': gaps[i - 1]['stop'], 'stop': end, 'duration': end - gaps[i - 1]['stop']})
        else:
            # Middle chunks: between the end of one gap and the start of the next gap
            chunk_list.append({'Type': 'chunk', 'start': gaps[i - 1]['stop'], 'stop': gaps[i]['start'], 'duration': gaps[i]['start'] - gaps[i - 1]['stop']})

    return gaps + chunk_list

In [24]:
gaps_n_chunks=find_gaps_and_chunks(data['Date'])  # Assuming 'Date' is the Series of interest

In [25]:
gaps_n_chunks

[{'Type': 'gap',
  'start': np.int64(1723244400),
  'stop': np.int64(1723420800),
  'duration': np.int64(176400)},
 {'Type': 'chunk',
  'start': np.int64(1722816000),
  'stop': np.int64(1723244400),
  'duration': np.int64(428400)},
 {'Type': 'chunk',
  'start': np.int64(1723420800),
  'stop': np.int64(1723849200),
  'duration': np.int64(428400)}]

In [26]:
gaps_n_chunks=sorted(gaps_n_chunks, key=lambda x: x['start'])
gaps_n_chunks

[{'Type': 'chunk',
  'start': np.int64(1722816000),
  'stop': np.int64(1723244400),
  'duration': np.int64(428400)},
 {'Type': 'gap',
  'start': np.int64(1723244400),
  'stop': np.int64(1723420800),
  'duration': np.int64(176400)},
 {'Type': 'chunk',
  'start': np.int64(1723420800),
  'stop': np.int64(1723849200),
  'duration': np.int64(428400)}]

In [27]:
(1723420800-1723244400)/60/60

49.0

I need to make a distinction between qualified gaps from things like weekends and trading hours vs unqualified gaps when there is just not data.<br>
<span style="color: green;">For now I will just do all gaps and work on adding gap qualification after regular gap removal works.</span>

In [114]:
data

,True_Time,Collapsed_Time,Time_Shift,Date,Open,High,Low,Close
Index,,,,,,,,
0,1722816000,0,0,1722816000,148.500000,152.486985,146.836262,150.000000
1,1722819600,0,0,1722819600,151.120940,152.965735,149.972488,150.035207
2,1722823200,0,0,1722823200,150.211239,156.323894,148.434026,152.428834
3,1722826800,0,0,1722826800,153.262769,158.166219,152.595520,157.529194
4,1722830400,0,0,1722830400,159.260089,161.889206,154.705345,157.264062
...,...,...,...,...,...,...,...,...
235,1723834800,0,0,1723834800,315.889426,321.240349,298.135907,304.166881
236,1723838400,0,0,1723838400,297.616982,322.119848,289.741502,318.055344
237,1723842000,0,0,1723842000,323.277861,329.806891,302.372442,306.592071


In [115]:
data[data['Date'] == gaps_n_chunks[2]['start']]

,True_Time,Collapsed_Time,Time_Shift,Date,Open,High,Low,Close
Index,,,,,,,,
120,1723420800,0,0,1723420800,207.611881,209.987215,206.328124,208.178163


In [116]:
gaps_n_chunks[0]

{'Type': 'chunk',
 'start': np.int64(1722816000),
 'stop': np.int64(1723244400),
 'duration': np.int64(428400)}

## function to align the chunks

In [28]:
def align_chunks(chunks: list[dict[str, int]], data: pd.DataFrame):
    for chunk in chunks:
        # if the start of the chunk is equal to the start of the dataframe then continue
        if chunk['start'] == data['Date'].min():
            continue
        
        start_index = data[data['Date'] == chunk['start']].index[0]
        stop_index = data[data['Date'] == chunk['stop']].index[0]

        # Set the time shift for the chunk
        data.loc[start_index:stop_index, 'Time_Shift'] = chunk['duration']



In [29]:
align_chunks(chunkos,data)

In [30]:
data

,True_Time,Collapsed_Time,Time_Shift,Date,Open,High,Low,Close
Index,,,,,,,,
0,1722816000,0,0,1722816000,148.500000,152.486985,146.836262,150.000000
1,1722819600,0,0,1722819600,151.120940,152.965735,149.972488,150.035207
2,1722823200,0,0,1722823200,150.211239,156.323894,148.434026,152.428834
3,1722826800,0,0,1722826800,153.262769,158.166219,152.595520,157.529194
4,1722830400,0,0,1722830400,159.260089,161.889206,154.705345,157.264062
...,...,...,...,...,...,...,...,...
235,1723834800,0,428400,1723834800,315.889426,321.240349,298.135907,304.166881
236,1723838400,0,428400,1723838400,297.616982,322.119848,289.741502,318.055344
237,1723842000,0,428400,1723842000,323.277861,329.806891,302.372442,306.592071


In [31]:
start_index = data[data['Date'] == gap['start']].index[0]
stop_index = data[data['Date'] == gap['stop']].index[0]

NameError: name 'gap' is not defined

# Market hours boolean arrays


<span style="color: red;">Assume that the data is correct and make generators that are false for overnight, premarket etc</span>

In [3]:
print(pytz.all_timezones)

['Africa/Abidjan', 'Africa/Accra', 'Africa/Addis_Ababa', 'Africa/Algiers', 'Africa/Asmara', 'Africa/Asmera', 'Africa/Bamako', 'Africa/Bangui', 'Africa/Banjul', 'Africa/Bissau', 'Africa/Blantyre', 'Africa/Brazzaville', 'Africa/Bujumbura', 'Africa/Cairo', 'Africa/Casablanca', 'Africa/Ceuta', 'Africa/Conakry', 'Africa/Dakar', 'Africa/Dar_es_Salaam', 'Africa/Djibouti', 'Africa/Douala', 'Africa/El_Aaiun', 'Africa/Freetown', 'Africa/Gaborone', 'Africa/Harare', 'Africa/Johannesburg', 'Africa/Juba', 'Africa/Kampala', 'Africa/Khartoum', 'Africa/Kigali', 'Africa/Kinshasa', 'Africa/Lagos', 'Africa/Libreville', 'Africa/Lome', 'Africa/Luanda', 'Africa/Lubumbashi', 'Africa/Lusaka', 'Africa/Malabo', 'Africa/Maputo', 'Africa/Maseru', 'Africa/Mbabane', 'Africa/Mogadishu', 'Africa/Monrovia', 'Africa/Nairobi', 'Africa/Ndjamena', 'Africa/Niamey', 'Africa/Nouakchott', 'Africa/Ouagadougou', 'Africa/Porto-Novo', 'Africa/Sao_Tome', 'Africa/Timbuktu', 'Africa/Tripoli', 'Africa/Tunis', 'Africa/Windhoek', 'Ameri

In [4]:
@lru_cache(maxsize=16)
def get_timezone(tz_str):
    return pytz.timezone(tz_str)

In [17]:
def is_weekday_mask(unix_times: np.ndarray, tz_str: str = "US/Eastern") -> np.ndarray:
    """
    Returns a boolean mask indicating which timestamps fall on weekdays (Monday-Friday) in the specified timezone.

    Parameters
    ----------
    unix_times : np.ndarray
        Array of UNIX timestamps (seconds since epoch).
    tz_str : str, optional
        Timezone string (e.g., "US/Eastern"). Default is "US/Eastern".

    Supported Time Step Durations
    ----------------------------
    - Hourly
    - Minutely
    - Daily

    The function works for UNIX timestamps representing hours, minutes, or days.
    For sub-minute or irregular intervals, ensure timestamps are valid and correspond to actual datetimes.

    Returns
    -------
    np.ndarray
        Boolean mask where True indicates the timestamp is a weekday (Monday-Friday).
    """
    tz = get_timezone(tz_str)
    weekdays = np.array([
        datetime.fromtimestamp(ts, tz).weekday() for ts in unix_times
    ])
    return (weekdays >= 0) & (weekdays <= 4)

In [ ]:
def is_market_hour_mask(
    unix_times: np.ndarray,
    market_open=9,
    market_close=16,
    tz_str: str = "US/Eastern"
) -> np.ndarray:
    """
    Returns a boolean mask for regular market hours (default: 9am-4pm US Eastern).
    tz_str: timezone string, e.g. "US/Eastern"


    Supported Time Step Durations
    ----------------------------
    - Hourly
    - Daily

    This function works for UNIX timestamps representing hours or days.
    For sub-hour or irregular intervals, ensure timestamps are valid and correspond to actual datetimes.
    It does not account for holidays.

    Returns
    -------
    np.ndarray
        Boolean mask where True indicates the timestamp is within market hours.
    """
    tz = get_timezone(tz_str)
    hours = np.array([
        datetime.fromtimestamp(ts, tz).hour for ts in unix_times
    ])
    return (hours >= market_open) & (hours < market_close)

In [18]:
def is_market_hour_minute_mask(
    unix_times: np.ndarray,
    market_open_hour=9,
    market_open_minute=30,
    market_close_hour=16,
    market_close_minute=0,
    tz_str: str = "US/Eastern"
) -> np.ndarray:
    """
    Returns a boolean mask for market hours with minute precision.
    Only True for times between market_open_hour:market_open_minute and market_close_hour:market_close_minute.
    tz_str: timezone string, e.g. "US/Eastern"

    Supported Time Step Durations
    ----------------------------
    - Hourly
    - Minutely
    - Daily

    This function works for UNIX timestamps representing hours, minutes, or days.
    For sub-minute or irregular intervals, ensure timestamps are valid and correspond to actual datetimes.

    Returns
    -------
    np.ndarray
        Boolean mask where True indicates the timestamp is within market hours.
    """
    tz = get_timezone(tz_str)
    times = np.array([
        datetime.fromtimestamp(ts, tz) for ts in unix_times
    ])
    opens = np.array([
        (dt.hour > market_open_hour or (dt.hour == market_open_hour and dt.minute >= market_open_minute))
        for dt in times
    ])
    closes = np.array([
        (dt.hour < market_close_hour or (dt.hour == market_close_hour and dt.minute < market_close_minute))
        for dt in times
    ])
    return opens

In [19]:
datetime(2024, 8, 5, 0, 0).isoformat()  # Monday

'2024-08-05T00:00:00'

In [20]:
int(datetime(2024, 8, 5, 0, 0).timestamp())

1722830400

In [21]:
# Example: generate some hourly unix timestamps for a week
start_dt = datetime(2024, 8, 5, 0, 0)  # Monday
unix_times = np.array([
    int((start_dt + pd.Timedelta(hours=i)).timestamp())
    for i in range(7 * 24)
])
len(unix_times)

168

In [25]:
# Test is_weekday_mask
weekday_mask = is_weekday_mask(unix_times, tz_str="US/Eastern")
print("Weekday mask (first week):", weekday_mask[:24*7])  # Show first week

# Test is_market_hour_mask
market_hour_mask = is_market_hour_mask(unix_times, market_open=9, market_close=16, tz_str="US/Eastern")
print("Market hour mask (first week):", market_hour_mask[:24*7])  # Show first week

# Combine both masks to get trading hours on weekdays
trading_mask = weekday_mask & market_hour_mask
print("Trading mask (first week):", trading_mask[:24*7])

# Show which datetimes are trading hours
for i in range(48):
    if trading_mask[i]:
        dt = datetime.fromtimestamp(unix_times[i], get_timezone("US/Eastern"))
        print(f"Trading hour: {dt}")

Weekday mask (first week): [ True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
 False False False False False False False False False False False False
 False False False False False False False False False False False False
 False False False False False False False False False False False False
 False False False False

# Apply gaps and chunks to masks

In [ ]:
# I think i am going to need to apply the gaps and chunks to the masks
# if i create these dictionaries with start and stop times, i can apply iloc operations to the dataframe
# more efficient than looping through every value and checking if it's in a gap or chunk

In [26]:
def find_blocks(mask: np.ndarray):
    """
    Finds contiguous blocks of True or False in a boolean mask.
    Returns a list of dicts: {'value': bool, 'start': int, 'stop': int, 'length': int}
    """
    mask = np.asarray(mask)
    # Find where the value changes
    changes = np.where(np.diff(mask) != 0)[0] + 1
    # Add start and end
    indices = np.concatenate(([0], changes, [len(mask)]))
    blocks = []
    for i in range(len(indices) - 1):
        val = mask[indices[i]]
        blocks.append({
            'value': bool(val),
            'start': indices[i],
            'stop': indices[i+1],
            'length': indices[i+1] - indices[i]
        })
    return blocks

# Example usage:
blocks = find_blocks(trading_mask)
for block in blocks:
    print(block)

{'value': False, 'start': np.int64(0), 'stop': np.int64(9), 'length': np.int64(9)}
{'value': True, 'start': np.int64(9), 'stop': np.int64(16), 'length': np.int64(7)}
{'value': False, 'start': np.int64(16), 'stop': np.int64(33), 'length': np.int64(17)}
{'value': True, 'start': np.int64(33), 'stop': np.int64(40), 'length': np.int64(7)}
{'value': False, 'start': np.int64(40), 'stop': np.int64(57), 'length': np.int64(17)}
{'value': True, 'start': np.int64(57), 'stop': np.int64(64), 'length': np.int64(7)}
{'value': False, 'start': np.int64(64), 'stop': np.int64(81), 'length': np.int64(17)}
{'value': True, 'start': np.int64(81), 'stop': np.int64(88), 'length': np.int64(7)}
{'value': False, 'start': np.int64(88), 'stop': np.int64(105), 'length': np.int64(17)}
{'value': True, 'start': np.int64(105), 'stop': np.int64(112), 'length': np.int64(7)}
{'value': False, 'start': np.int64(112), 'stop': np.int64(168), 'length': np.int64(56)}
